In [2]:
# Désinstaller l'ancien package
#!pip uninstall -y pinecone-client pinecone

# Installer le nouveau package officiel
#!pip install -U pinecone --no-cache-dir

print("✅ Pinecone installed!")

✅ Pinecone installed!


In [3]:
# Setup Pinecone (après restart)
import os
from google.colab import userdata
from pinecone import Pinecone

# Load API key from secrets
api_key = userdata.get('PINECONE_API_KEY')
os.environ["PINECONE_API_KEY"] = api_key

# Initialize Pinecone client
pc = Pinecone(api_key=api_key)

print("✅ Pinecone client initialized!")

✅ Pinecone client initialized!


In [4]:
# 1.1: Define query and documents
query = "Tell me about Apple's products"

documents = [
    "An apple is a sweet, edible fruit produced by an apple tree. Apples are rich in fiber and vitamin C.",  # Fruit document
    "Apple Inc. is an American technology company that makes iPhones, iPads, MacBooks, and Apple Watches.",  # Company products
    "Oranges and apples are both popular fruits. Apples can be red, green, or yellow.",  # Another fruit document
    "Apple released the new iPhone 15 with advanced camera features and A17 chip technology.",  # Another company document
    "Granny Smith apples are known for their tart flavor and are great for baking apple pies."  # One more document
]

print(f"Query: '{query}'")
print(f"\n📄 Original documents ({len(documents)} total):")
for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc[:70]}...")

Query: 'Tell me about Apple's products'

📄 Original documents (5 total):
1. An apple is a sweet, edible fruit produced by an apple tree. Apples ar...
2. Apple Inc. is an American technology company that makes iPhones, iPads...
3. Oranges and apples are both popular fruits. Apples can be red, green, ...
4. Apple released the new iPhone 15 with advanced camera features and A17...
5. Granny Smith apples are known for their tart flavor and are great for ...


In [5]:
# 1.2: Call the reranker
from pinecone import RerankModel

print("\n🔄 Reranking documents...")

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3  # Get top 3 most relevant results
)

print("✅ Reranking complete!")


🔄 Reranking documents...
✅ Reranking complete!


In [6]:
# 1.3: Inspect reranked results
def show_reranked_results(query, matches):
    print(f"Query: '{query}'")
    print("\n🏆 Reranked Results:")
    for i, m in enumerate(matches):
        print(f"\n{i+1}. Score: {m.score:.4f}")  # m.score contains the relevance score
        print(f"   ID: {m.document.id}")
        print(f"   Text: {m.document.text}")  # m.document.text contains the document text

show_reranked_results(query, reranked.data)  # reranked.data contains the matches

Query: 'Tell me about Apple's products'

🏆 Reranked Results:

1. Score: 0.8630
   ID: 1
   Text: Apple Inc. is an American technology company that makes iPhones, iPads, MacBooks, and Apple Watches.

2. Score: 0.0969
   ID: 0
   Text: An apple is a sweet, edible fruit produced by an apple tree. Apples are rich in fiber and vitamin C.

3. Score: 0.0777
   ID: 3
   Text: Apple released the new iPhone 15 with advanced camera features and A17 chip technology.


In [7]:
# 2.1: Import modules and define environment settings
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings
cloud = os.getenv('PINECONE_CLOUD', 'aws')  # Default cloud provider
region = os.getenv('PINECONE_REGION', 'us-east-1')  # Default region

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index'  # Name for our medical notes index

print(f"☁️  Cloud: {cloud}")
print(f"🌍 Region: {region}")
print(f"📇 Index: {index_name}")

☁️  Cloud: aws
🌍 Region: us-east-1
📇 Index: medical-notes-index


In [8]:
# 2.2: Create or recreate the index

# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)
    print(f"🗑️  Deleted existing index '{index_name}'")

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384,  # Matches our embedding model size (all-MiniLM-L6-v2 outputs 384 dims)
    metric='cosine',  # Cosine similarity is best for text embeddings
    spec=spec
)

print(f"✅ Created index '{index_name}'")
print(f"   Dimension: 384")
print(f"   Metric: cosine")

✅ Created index 'medical-notes-index'
   Dimension: 384
   Metric: cosine


In [9]:
# 3.1: Download and read JSONL data
import requests
import tempfile

print("📥 Downloading medical notes data from GitHub...")

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from GitHub
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

print("✅ Data downloaded and loaded successfully!")

📥 Downloading medical notes data from GitHub...
✅ Data downloaded and loaded successfully!


In [10]:
# 3.2: Preview the DataFrame
print("Data shape:", df.shape)  # df.shape shows (rows, columns)
print(f"\n📊 Dataset contains {df.shape[0]} medical notes with {df.shape[1]} columns")
print("\n📋 Column names:", list(df.columns))
print("\n🔍 First few rows:")
df.head()

Data shape: (100, 3)

📊 Dataset contains 100 medical notes with 3 columns

📋 Column names: ['id', 'values', 'metadata']

🔍 First few rows:


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


In [11]:
# 4.1: Instantiate index client and upsert

# Instantiate an index client
index = pc.Index(name=index_name)

print(f"📤 Upserting {len(df)} vectors into index '{index_name}'...")

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df)  # Pass the DataFrame variable

print("✅ Data upserted successfully!")

📤 Upserting 100 vectors into index 'medical-notes-index'...


sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

✅ Data upserted successfully!


In [12]:
# 4.2: Wait for availability

def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: {vector_count}")
    return vector_count > 0  # Should be greater than 0 (at least 1 vector)

print("⏳ Waiting for index to be ready...")
while not is_fresh(index):
    time.sleep(5)

print("\n✅ Index ready!")
print("\n📊 Final index statistics:")
stats = index.describe_index_stats()
print(f"   Total vectors: {stats.total_vector_count}")

⏳ Waiting for index to be ready...
Vector count: 100

✅ Index ready!

📊 Final index statistics:
   Total vectors: 100


In [13]:
# 5.1: Define embedding function

def get_embedding(input_question):
    """Convert text to embedding vector"""
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        model_output = model(**encoded_input)
        embedding = model_output.last_hidden_state[0].mean(dim=0)  # Average over dimension 0 (sequence length)

    return embedding

print("✅ Embedding function defined!")

✅ Embedding function defined!


In [14]:
# 5.2: Run a semantic search query

# Build a query to search
question = "patient has severe chest pain and difficulty breathing"  # Medical question
query = get_embedding(question).tolist()

print(f"🔍 Question: '{question}'")
print(f"📏 Query vector dimension: {len(query)}")

# Get results
results = index.query(vector=[query], top_k=5, include_metadata=True)  # Get top 5 results

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

print(f"\n✅ Found {len(sorted_matches)} relevant medical notes")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

🔍 Question: 'patient has severe chest pain and difficulty breathing'
📏 Query vector dimension: 384

✅ Found 5 relevant medical notes


In [15]:
# 6.1: Display initial search results

def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\n🔍 Initial Search Results:')
    for i, match in enumerate(matches):
        print(f'\n{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f'      Score: {match["score"]:.4f}')  # match["score"] contains the similarity score
        print(f'      Metadata: {match["metadata"]}')  # match["metadata"] contains metadata

show_results(question, sorted_matches)

Question: 'patient has severe chest pain and difficulty breathing'

🔍 Initial Search Results:

   1. ID: P001
      Score: 0.6797
      Metadata: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

   2. ID: P003
      Score: 0.4775
      Metadata: {'diagnosis': 'bronchitis', 'symptoms': 'cough, wheezing', 'treatment': 'antibiotics'}

   3. ID: P016
      Score: 0.4677
      Metadata: {'condition': 'heart murmur', 'referral': 'cardiology'}

   4. ID: P063
      Score: 0.4395
      Metadata: {'diagnosis': 'pneumonia', 'symptoms': 'cough, fever', 'treatment': 'antibiotics'}

   5. ID: P032
      Score: 0.4348
      Metadata: {'condition': 'asthma', 'treatment': 'nebulizer'}


In [16]:
# 6.2: Prepare documents for reranking

# Create documents with concatenated metadata field as "reranking_field"
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])  # match['metadata'] contains the metadata
    }
    for match in results['matches']
]

print(f"✅ Prepared {len(transformed_documents)} documents for reranking")
print("\n📄 Example transformed document:")
import json
print(json.dumps(transformed_documents[0], indent=2))

✅ Prepared 5 documents for reranking

📄 Example transformed document:
{
  "id": "P001",
  "reranking_field": "symptoms: chest pain; tests: EKG, stress test"
}


In [17]:
# 6.3: Execute serverless reranking

# Define a more specific query for reranking
refined_query = "patient with acute myocardial infarction needs urgent surgical intervention"  # More specific medical question

print(f"🔄 Reranking with refined query: '{refined_query}'")

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,  # Get top 3 reranked results
    return_documents=True,
)

print("✅ Reranking complete!")

🔄 Reranking with refined query: 'patient with acute myocardial infarction needs urgent surgical intervention'
✅ Reranking complete!


In [18]:
# 6.4: Show reranked results

def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\n🏆 Reranked Results:')
    for i, match in enumerate(matches):
        print(f'\n{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f'      Score: {match.score:.4f}')  # match.score contains the reranking score
        print(f'      Reranking Field: {match.document.reranking_field}')  # match.document.reranking_field contains the searchable field

show_reranked_results(refined_query, reranked_results.data)  # reranked_results.data contains the results

Question: 'patient with acute myocardial infarction needs urgent surgical intervention'

🏆 Reranked Results:

   1. ID: P001
      Score: 0.0352
      Reranking Field: symptoms: chest pain; tests: EKG, stress test

   2. ID: P016
      Score: 0.0025
      Reranking Field: condition: heart murmur; referral: cardiology

   3. ID: P003
      Score: 0.0000
      Reranking Field: diagnosis: bronchitis; symptoms: cough, wheezing; treatment: antibiotics


In [19]:
# 6.5: Comparison - Before vs After Reranking

print("="*80)
print("📊 COMPARISON: Original Search vs Reranked Results")
print("="*80)

print("\n🔍 ORIGINAL SEARCH RESULTS (top 3):")
for i, match in enumerate(sorted_matches[:3], 1):
    print(f"\n{i}. ID: {match['id']} | Score: {match['score']:.4f}")
    print(f"   Metadata: {match['metadata']}")

print("\n" + "="*80)
print("\n🏆 RERANKED RESULTS (top 3):")
for i, match in enumerate(reranked_results.data, 1):
    print(f"\n{i}. ID: {match.document.id} | Score: {match.score:.4f}")
    print(f"   Field: {match.document.reranking_field}")

print("\n" + "="*80)
print("💡 Notice how reranking reorders results based on the refined query!")

📊 COMPARISON: Original Search vs Reranked Results

🔍 ORIGINAL SEARCH RESULTS (top 3):

1. ID: P001 | Score: 0.6797
   Metadata: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

2. ID: P003 | Score: 0.4775
   Metadata: {'diagnosis': 'bronchitis', 'symptoms': 'cough, wheezing', 'treatment': 'antibiotics'}

3. ID: P016 | Score: 0.4677
   Metadata: {'condition': 'heart murmur', 'referral': 'cardiology'}


🏆 RERANKED RESULTS (top 3):

1. ID: P001 | Score: 0.0352
   Field: symptoms: chest pain; tests: EKG, stress test

2. ID: P016 | Score: 0.0025
   Field: condition: heart murmur; referral: cardiology

3. ID: P003 | Score: 0.0000
   Field: diagnosis: bronchitis; symptoms: cough, wheezing; treatment: antibiotics

💡 Notice how reranking reorders results based on the refined query!


In [20]:
# 6.6: Clean up (optional - run when done)

# Delete the index to save resources
# pc.delete_index(name=index_name)
# print(f"🗑️  Deleted index '{index_name}'")

print("\n⚠️  Uncomment the code above to delete the index when you're done")
print("💰 This helps avoid unnecessary charges on your Pinecone account")


⚠️  Uncomment the code above to delete the index when you're done
💰 This helps avoid unnecessary charges on your Pinecone account


In [21]:
# Complete RAG pipeline demonstration

def complete_rag_pipeline(question, index, pc, top_k=5, rerank_top_n=3):
    """
    Full RAG pipeline: Query → Retrieve → Rerank → Display
    """
    print("="*80)
    print(f"❓ Question: '{question}'")
    print("="*80)

    # Step 1: Get embedding
    print("\n📊 Step 1: Converting question to embedding...")
    query_embedding = get_embedding(question).tolist()

    # Step 2: Search index
    print(f"🔍 Step 2: Searching index for top {top_k} results...")
    results = index.query(vector=[query_embedding], top_k=top_k, include_metadata=True)
    sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

    print(f"   Found {len(sorted_matches)} results")

    # Step 3: Prepare for reranking
    print("📋 Step 3: Preparing documents for reranking...")
    transformed_docs = [
        {
            'id': match['id'],
            'reranking_field': '; '.join([f"{k}: {v}" for k, v in match['metadata'].items()])
        }
        for match in results['matches']
    ]

    # Step 4: Rerank
    print(f"🔄 Step 4: Reranking to get top {rerank_top_n}...")
    reranked = pc.inference.rerank(
        model="bge-reranker-v2-m3",
        query=question,
        documents=transformed_docs,
        rank_fields=["reranking_field"],
        top_n=rerank_top_n,
        return_documents=True,
    )

    # Step 5: Display results
    print("\n✨ Final Reranked Results:")
    for i, match in enumerate(reranked.data, 1):
        print(f"\n{i}. Score: {match.score:.4f}")
        print(f"   ID: {match.document.id}")
        print(f"   Content: {match.document.reranking_field[:150]}...")

    return reranked

# Test with multiple medical questions
test_questions = [
    "patient with diabetes needs insulin management",
    "broken bone requires surgical treatment",
    "child with high fever and cough symptoms"
]

for q in test_questions:
    complete_rag_pipeline(q, index, pc, top_k=5, rerank_top_n=3)
    print("\n")

❓ Question: 'patient with diabetes needs insulin management'

📊 Step 1: Converting question to embedding...
🔍 Step 2: Searching index for top 5 results...
   Found 5 results
📋 Step 3: Preparing documents for reranking...
🔄 Step 4: Reranking to get top 3...

✨ Final Reranked Results:

1. Score: 0.1298
   ID: P014
   Content: blood_sugar: 8.0; condition: diabetes; medications: insulin...

2. Score: 0.0207
   ID: P098
   Content: blood_sugar: 7.0; condition: diabetes; medications: adjusted...

3. Score: 0.0205
   ID: P050
   Content: blood_sugar: 7.0; condition: diabetes; medications: adjusted...


❓ Question: 'broken bone requires surgical treatment'

📊 Step 1: Converting question to embedding...
🔍 Step 2: Searching index for top 5 results...
   Found 5 results
📋 Step 3: Preparing documents for reranking...
🔄 Step 4: Reranking to get top 3...

✨ Final Reranked Results:

1. Score: 0.0281
   ID: P007
   Content: surgery: knee arthroscopy; symptoms: pain, swelling; treatment: physical thera